# 01 · Audit the Twi data
Run after `00_verify_resources`. Reports usable hours per speaker under duration caps, recording quality per speaker
(SNR-like gap, clipping, silence, pitch, warning flags), characters the MMS vocabulary lacks, and leftover digits, and
saves a few clips per speaker. **Paste the JSON back into the chat, and tell us which
speaker sounds best and whether the clips are Asante or Akuapem Twi.**

In [ ]:
# Fresh clone, or update an existing one
!test -d /content/tts && git -C /content/tts pull -q || git clone -q https://github.com/YCienne/TTS.git /content/tts
%cd /content/tts
!pip install -q -r requirements.txt

In [ ]:
!python scripts/audit_data.py --out reports/audit.json

In [ ]:
# One-line-per-speaker summary (train): paste this table back as well
import json, pandas as pd
sp = json.load(open("reports/audit.json"))["train"]["speakers"]
pd.DataFrame([{
    "speaker": k, "gender": v["gender"], "hours": v["hours"],
    "h<=15s": v["usable"]["<=15s"]["hours"], "h<=20s": v["usable"]["<=20s"]["hours"], "h<=30s": v["usable"]["<=30s"]["hours"],
    "snr_db": v["quality"]["snr_db_median"], "clipped_clips": v["quality"]["clips_with_clipping"],
    "f0_hz": v["quality"].get("f0_median_hz"), "chars/s": v["chars_per_sec_median"],
    "flags": "; ".join(v["quality"]["flags"]) or "-",
} for k, v in sp.items()]).set_index("speaker")

In [ ]:
# Listen: three clips per speaker, with their transcripts
import json, IPython.display as ipd
for s in json.load(open("reports/audit.json"))["samples"]:
    print(f"speaker {s['speaker']} ({s['gender']}): {s['text']}")
    display(ipd.Audio(s["file"]))